# Hãy nâng cấp lên trình độ CHUYÊN NGHIỆP!

Các kỹ thuật RAG nâng cao!

Hãy bắt đầu bằng cách tìm hiểu sâu về quá trình nạp dữ liệu:

1. Không dùng LangChain! Chỉ dùng các công cụ gốc để có độ linh hoạt tối đa
2. Hãy dùng một LLM để chia các đoạn một cách hợp lý
3. Hãy dùng kích thước đoạn và bộ mã hóa tốt nhất từ hôm qua
4. Hãy để LLM viết lại các đoạn theo cách hữu ích nhất ("tiền xử lý tài liệu")

In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [ ]:
# Lấy cảm hứng từ Document của LangChain — hãy tạo một lớp tương tự

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# Một lớp biểu diễn đoạn văn bản một cách đầy đủ

class Chunk(BaseModel):
    headline: str = Field(description="Tiêu đề ngắn gọn cho đoạn này, thường chỉ vài từ và có khả năng cao nhất xuất hiện trong một truy vấn")
    summary: str = Field(description="Một vài câu tóm tắt nội dung của đoạn này để trả lời các câu hỏi thường gặp")
    original_text: str = Field(description="Nguyên văn của đoạn này từ tài liệu được cung cấp, giữ nguyên tuyệt đối, không thay đổi dưới bất kỳ hình thức nào")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Ba bước:

1. Lấy tài liệu từ cơ sở tri thức, giống như LangChain đã làm
2. Gọi một LLM để chuyển tài liệu thành các đoạn
3. Lưu các đoạn vào Chroma

Chỉ vậy thôi!

### Hãy bắt đầu với Bước 1

In [ ]:
def fetch_documents():
    """Phiên bản tự xây dựng của LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Đã nạp {len(documents)} tài liệu")
    return documents

In [ ]:
documents = fetch_documents()

### Xong rồi! Chuyển sang Bước 2 — tạo các đoạn

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
Bạn nhận một tài liệu và chia tài liệu đó thành các đoạn chồng lấn cho một Cơ sở tri thức.

Tài liệu đến từ ổ đĩa dùng chung của một công ty có tên Insurellm.
Loại tài liệu: {document["type"]}
Tài liệu được lấy từ: {document["source"]}

Một chatbot sẽ sử dụng các đoạn này để trả lời câu hỏi về công ty.
Bạn nên chia tài liệu theo cách phù hợp, đồng thời bảo đảm toàn bộ tài liệu được trả về trong các đoạn — không được bỏ sót bất kỳ nội dung nào.
Tài liệu này có lẽ nên được chia thành {how_many} đoạn, nhưng bạn có thể dùng nhiều hơn hoặc ít hơn nếu thấy phù hợp.
Các đoạn nên chồng lấn ở mức hợp lý; thông thường là khoảng 25% hoặc khoảng 50 từ, để cùng một nội dung xuất hiện trong nhiều đoạn nhằm mang lại kết quả truy xuất tốt nhất.

Với mỗi đoạn, bạn cần cung cấp một tiêu đề, một phần tóm tắt và nguyên văn của đoạn đó.
Khi kết hợp lại, các đoạn của bạn phải đại diện cho toàn bộ tài liệu và có phần chồng lấn.

Đây là tài liệu:

{document["text"]}

Hãy trả lời bằng các đoạn đã chia.
"""

In [ ]:
print(make_prompt(documents[0]))

In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
make_messages(documents[0])

In [ ]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
chunks = create_chunks(documents)

In [ ]:
print(len(chunks))

### Thật dễ dàng! Tuy có hơi chậm một chút.

Trong phiên bản mô-đun Python, tôi đã khéo léo dùng Pool đa tiến trình để chạy song song,
nhưng nếu bạn gặp Lỗi giới hạn tốc độ thì có thể tắt tính năng này trong mã.

### Cuối cùng, Bước 3 — lưu các embedding

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Kho vectơ đã được tạo với {collection.count()} tài liệu")

In [ ]:
create_embeddings(chunks)

# Không còn gì phải làm ở đây nữa... đúng không?

Khoan đã! Bạn nghĩ tôi quên rồi sao??

In [ ]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Tạo biểu đồ phân tán 2D
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='Trực quan hóa kho vectơ Chroma trong không gian 2D',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Tạo biểu đồ phân tán 3D
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='Trực quan hóa kho vectơ Chroma trong không gian 3D',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## Và bây giờ — hãy xây dựng một hệ thống RAG nâng cao!

Chúng ta sẽ sử dụng các kỹ thuật sau:

1. Xếp hạng lại — sắp xếp lại thứ tự các kết quả
2. Viết lại truy vấn

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="Thứ tự liên quan của các đoạn, từ liên quan nhất đến ít liên quan nhất, dựa theo số id của đoạn"
    )

In [ ]:
def rerank(question, chunks):
    system_prompt = """
Bạn là một hệ thống xếp hạng lại tài liệu.
Bạn được cung cấp một câu hỏi và danh sách các đoạn văn bản liên quan từ kết quả truy vấn một cơ sở tri thức.
Các đoạn được cung cấp theo thứ tự truy xuất; thứ tự này gần đúng theo mức độ liên quan, nhưng bạn có thể cải thiện nó.
Bạn phải xếp hạng các đoạn được cung cấp theo mức độ liên quan đến câu hỏi, với đoạn liên quan nhất đứng đầu.
Chỉ trả lời bằng danh sách id của các đoạn theo thứ tự đã xếp hạng, không thêm bất kỳ nội dung nào khác. Phải bao gồm tất cả id của các đoạn được cung cấp sau khi xếp hạng lại.
"""
    user_prompt = f"Người dùng đã hỏi câu hỏi sau:\n\n{question}\n\nHãy sắp xếp tất cả các đoạn văn bản theo mức độ liên quan đến câu hỏi, từ liên quan nhất đến ít liên quan nhất. Phải bao gồm tất cả id của các đoạn được cung cấp sau khi xếp hạng lại.\n\n"
    user_prompt += "Đây là các đoạn:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# ID ĐOẠN: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Chỉ trả lời bằng danh sách id của các đoạn theo thứ tự đã xếp hạng, không thêm bất kỳ nội dung nào khác."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
question = "Ai đã giành giải IIOTY?"
chunks = fetch_context_unranked(question)

In [ ]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
question = "Ai đã học tại Đại học Manchester?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked[0].page_content

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
Bạn là một trợ lý am hiểu và thân thiện, đại diện cho công ty Insurellm.
Bạn đang trò chuyện với người dùng về Insurellm.
Câu trả lời của bạn sẽ được đánh giá về độ chính xác, mức độ liên quan và tính đầy đủ, vì vậy hãy bảo đảm câu trả lời chỉ tập trung vào câu hỏi và trả lời câu hỏi một cách trọn vẹn.
Nếu không biết câu trả lời, hãy nói rõ điều đó.
Để làm ngữ cảnh, dưới đây là các trích đoạn cụ thể từ Cơ sở tri thức có thể liên quan trực tiếp đến câu hỏi của người dùng:
{context}

Dựa trên ngữ cảnh này, hãy trả lời câu hỏi của người dùng. Hãy chính xác, liên quan và đầy đủ.
"""

In [ ]:
# Trong ngữ cảnh, hãy bao gồm nguồn của đoạn

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Trích từ {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Viết lại câu hỏi của người dùng thành một câu hỏi cụ thể hơn, có khả năng tìm ra nội dung liên quan trong Cơ sở tri thức cao hơn."""
    message = f"""
Bạn đang trò chuyện với một người dùng và trả lời các câu hỏi về công ty Insurellm.
Bạn sắp tra cứu thông tin trong một Cơ sở tri thức để trả lời câu hỏi của người dùng.

Đây là lịch sử cuộc trò chuyện của bạn với người dùng cho đến thời điểm hiện tại:
{history}

Và đây là câu hỏi hiện tại của người dùng:
{question}

Chỉ trả lời bằng một câu hỏi duy nhất đã được tinh chỉnh để bạn dùng khi tìm kiếm trong Cơ sở tri thức.
Đó phải là một câu hỏi cụ thể, RẤT ngắn gọn và có khả năng tìm ra nội dung liên quan cao nhất. Hãy tập trung vào các chi tiết của câu hỏi.
Không đề cập tên công ty trừ khi đó là một câu hỏi chung về công ty.
QUAN TRỌNG: CHỈ trả lời bằng truy vấn dành cho cơ sở tri thức, không thêm bất kỳ nội dung nào khác.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Ai đã giành giải IIOTY?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Trả lời câu hỏi bằng RAG, đồng thời trả về câu trả lời và ngữ cảnh đã truy xuất
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Ai đã giành giải IIOTY?", [])

In [ ]:
answer_question("Ai đã học tại Đại học Manchester?", [])